In [3]:
# custom imports
import nekt
import os                       # remove to production
from dotenv import load_dotenv  # remove to production
from pathlib import Path
from typing import Optional 
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# get Nekt data access token (remove to production)
load_dotenv()
nekt.data_access_token = os.getenv("NEKT_DATA_ACCESS_TOKEN")

# auxiliary functions
def load_nekt_table(table_name: str, layer_name: str) -> DataFrame:
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def display_df(df: DataFrame, limit=10):
    return df.limit(limit).toPandas()

def save_nekt_table(
    df: DataFrame, 
    layer_name: str, 
    table_name: str,
    folder_name: Optional[str] = None 
):
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )

# load your source tables (you can load multiple tables)
df_bronze_users        = load_nekt_table("users"        , "Bronze")
df_bronze_spaces       = load_nekt_table("spaces"       , "Bronze")
df_bronze_time_entries = load_nekt_table("time_entries" , "Bronze")

# filter columns
df_silver_users = df_bronze_users.select(
      F.col("id").cast("integer").alias("user_id")
    , F.col("username").alias("user_name")
)

df_silver_spaces = df_bronze_spaces.select(
      F.col("id").cast("long").alias("space_id")
    , F.col("name").alias("space_name")
)

df_silver_time_entries = df_bronze_time_entries.select(
      F.col("id").cast("long").alias("interval_id")
    , F.col("wid").cast("integer").alias("team_id")
    , F.col("user.id").cast("integer").alias("user_id")
    , F.col("user.username").alias("user_name")
    , F.col("task_location.space_id").cast("long").alias("space_id")
    , F.col("task.id").alias("task_id")
    , F.col("start").cast("long").alias("interval_date_start_ms")
    , F.to_timestamp(F.col("start") / 1000).alias("interval_date_start_iso")
    , F.col("end").cast("long").alias("interval_date_end_ms")
    , F.to_timestamp(F.col("end") / 1000).alias("interval_date_end_iso")
    , F.col("at").cast("long").alias("interval_date_added_ms")
    , F.to_timestamp(F.col("at") / 1000).alias("interval_date_added_iso")
)

display_df(df_silver_time_entries)

# save your results to one or more output tables
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

params: {'include_layer_database_name': 'true'}
Table details: {'id': '7adcd28a-c0e3-4a6a-aea2-1eeef140cee7', 'name': 'users', 'slug': 'users', 'database_id': 'users', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.users', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'deleted': False, 'created_at': '2026-02-26T11:33:03.805939-03:00', 'updated_at': '2026-03-02T15:28:42.292806-03:00'}
params: {'include_layer_database_name': 'true'}
Table details: {'id': '5169a424-7ea2-43be-98e5-f26828090499', 'name': 'spaces', 'slug': 'spaces', 'database_id': 'spaces', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.spaces', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'delet

,interval_id,team_id,user_id,user_name,space_id,task_id,interval_date_start_ms,interval_date_start_iso,interval_date_end_ms,interval_date_end_iso,interval_date_added_ms,interval_date_added_iso
0,4977451708571189070,2394683,18900326,Giovanna Buzo,90131730567,86afvcj6q,1772471657272,2026-03-02 17:14:17.272,1772475377272,2026-03-02 18:16:17.272,1772475380603,2026-03-02 18:16:20.603
1,4912481052197121252,2394683,3086411,Henrique Gala,90137646457,86aeknxm5,1768602403367,2026-01-16 22:26:43.367,1768602823367,2026-01-16 22:33:43.367,1768602827710,2026-01-16 22:33:47.710
2,4912480084369861843,2394683,3086411,Henrique Gala,90137646457,86aeknxm5,1768600004326,2026-01-16 21:46:44.326,1768602764326,2026-01-16 22:32:44.326,1768602770024,2026-01-16 22:32:50.024
3,4900945879701900169,2394683,3086411,Henrique Gala,90138864609,86aed0aec,1767909634992,2026-01-08 22:00:34.992,1767915274992,2026-01-08 23:34:34.992,1767915277850,2026-01-08 23:34:37.850
4,4906643907553258283,2394683,3086411,Henrique Gala,90138864609,86aed0aec,1768252562228,2026-01-12 21:16:02.228,1768254902228,2026-01-12 21:55:02.228,1768254906774,2026-01-12 21:55:06.774
5,4912480448351562969,2394683,3086411,Henrique Gala,90131730567,86a80xue8,1768602487371,2026-01-16 22:28:07.371,1768602787371,2026-01-16 22:33:07.371,1768602791716,2026-01-16 22:33:11.716
6,4912481516489796850,2394683,3086411,Henrique Gala,90131730567,86a80xue8,1768602071041,2026-01-16 22:21:11.041,1768602851041,2026-01-16 22:34:11.041,1768602855384,2026-01-16 22:34:15.384
7,4912472876122893356,2394683,3086411,Henrique Gala,90131730567,86a80xue8,1768599815847,2026-01-16 21:43:35.847,1768602335847,2026-01-16 22:25:35.847,1768602340373,2026-01-16 22:25:40.373
8,4970553605855367091,2394683,3086411,Henrique Gala,90138864609,86afp285v,1772063916674,2026-02-25 23:58:36.674,1772064216674,2026-02-26 00:03:36.674,1772064221639,2026-02-26 00:03:41.639
9,4921295791618516625,2394683,3086411,Henrique Gala,90131730567,86aemczmu,1769123180545,2026-01-22 23:06:20.545,1769128220545,2026-01-23 00:30:20.545,1769128227121,2026-01-23 00:30:27.121
